In [2]:
"""
Quick Missing Values Analysis — Module 2 Only
Prints results directly to console with example rows
"""

import glob
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# ── Environment detection (Colab vs Local) ────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import os as _os
    _os.listdir('/content/drive/MyDrive')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Exports')
    BASE_DIR = Path('/content/drive/MyDrive/Dashboard-BR-CA-Data')
    IN_COLAB = True
except Exception:
    BASE_DIR = Path.cwd().parent / 'Dataset'
    IN_COLAB = False

EXPORTS_DIR  = BASE_DIR / 'Exports'
NORM_FILE    = BASE_DIR / 'Normalised' / 'prices_normalised.csv'
REPORTS_DIR  = BASE_DIR / 'Reports'
LOOKER_DIR   = BASE_DIR / 'Looker'

print(f'Environment : {"Colab" if IN_COLAB else "Local"}')
print(f'Base dir    : {BASE_DIR}')
print(f'Exists      : {BASE_DIR.exists()}')


# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
KEY_COLS = ["Period", "Commodity", "Province", "Country"]

# ──────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────
def load_raw(filepath):
    df = pd.read_csv(filepath, skiprows=1, dtype=str)
    df.columns = df.columns.str.strip()
    
    # Remove footer lines (metadata at bottom of file)
    if "Period" in df.columns:
        # Step 1: Remove rows where Period contains metadata keywords
        metadata_keywords = ["source:", "note:", "notes:", "data source:", "disclaimer:", 
                            "citation:", "data extracted:", "extracted:", "date:", 
                            "© ", "copyright", "confidential"]
        period_lower = df["Period"].fillna("").astype(str).str.lower().str.strip()
        mask = period_lower.str.startswith(tuple(metadata_keywords))
        df = df[~mask]
        
        # Step 2: Remove rows where Period is not a valid date format
        period_values = df["Period"].fillna("").astype(str).str.strip()
        valid_mask = (
            (period_values == "") |  # Empty is ok (will be caught as missing value)
            (period_values.str.match(r'^\d{4}-\d{2}-\d{2}')) |  # YYYY-MM-DD
            (period_values.str.match(r'^\d{4}/\d{2}/\d{2}')) |  # YYYY/MM/DD
            (period_values.str.match(r'^\d{2}-\d{2}-\d{4}')) |  # DD-MM-YYYY
            (pd.to_datetime(df["Period"], errors='coerce').notna())  # Any parseable date
        )
        df = df[valid_mask]
        
        # Step 3: Remove trailing completely empty rows
        last_valid_idx = None
        for idx in reversed(df.index):
            row = df.loc[idx]
            has_data = row.notna().any() and (row.astype(str).str.strip() != "").any()
            if has_data:
                last_valid_idx = idx
                break
        
        if last_valid_idx is not None:
            df = df.loc[:last_valid_idx]
    
    df["_source_file"] = Path(filepath).name
    return df

def load_all(files):
    frames = []
    errors = []
    for f in files:
            frames.append(load_raw(f))
        except Exception as e:
            errors.append({"file": Path(f).name, "error": str(e)})
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return combined, errors

# ──────────────────────────────────────────────────────────────
# MODULE 2 — MISSING VALUES
# ──────────────────────────────────────────────────────────────
def module_missing(raw):
    df = raw.drop(columns=["_source_file"], errors="ignore")
    n  = len(df)
    
    # Detect missing: NaN or empty string
    col_missing = df.isna() | df.apply(lambda col: col.astype(str).str.strip() == "")
    
    results = {
        "per_column": [],
        "per_row_summary": [],
        "key_identifiers": [],
        "patterns": [],
        "examples": {}  # Store example rows
    }
    
    # ═══════════════════════════════════════════════════════════
    # 1. PER COLUMN
    # ═══════════════════════════════════════════════════════════
    col_pct = (col_missing.sum() / n * 100).round(2)
    
    for col in df.columns:
        missing_count = int(col_missing[col].sum())
        missing_pct = col_pct[col]
        
        # Determine if this is a critical column
        is_key_col = col in KEY_COLS or col in ["Value ($)", "Quantity"]
        is_medium_col = col == "Unit of measure"
        
        # Assign priority based on BOTH importance AND actual missing %
        if missing_pct == 0:
            # No missing data = no priority issue
            priority = "✅ NONE"
            status = "✅ OK"
        elif is_key_col:
            # Key columns: priority scales with missing %
            if missing_pct > 10:
                priority = "🔴 CRITICAL"
                status = "❌ FAIL"
            elif missing_pct > 5:
                priority = "🔴 HIGH"
                status = "❌ FAIL"
            elif missing_pct > 0:
                priority = "🟡 MEDIUM"
                status = "⚠️ WARN"
        elif is_medium_col:
            # Unit of measure
            if missing_pct > 20:
                priority = "🟡 HIGH"
                status = "⚠️ WARN"
            elif missing_pct > 5:
                priority = "🟡 MEDIUM"
                status = "⚠️ WARN"
            elif missing_pct > 0:
                priority = "🟢 LOW"
                status = "⚠️ WARN"
        else:
            # Non-critical columns (like State)
            if missing_pct > 50:
                priority = "🟡 MEDIUM"
                status = "⚠️ WARN"
            elif missing_pct > 20:
                priority = "🟢 LOW"
                status = "⚠️ WARN"
            elif missing_pct > 0:
                priority = "🟢 MINIMAL"
                status = "⚠️ WARN"
        
        # Assign action based on priority
        if priority in ["🔴 CRITICAL", "🔴 HIGH"]:
            action = "Investigate immediately — critical field"
        elif priority in ["🟡 HIGH", "🟡 MEDIUM"]:
            action = "Review and determine if imputation or removal needed"
        elif priority in ["🟢 LOW", "🟢 MINIMAL"]:
            action = "Optional — low priority cleanup"
        else:
            action = "None"
        
        results["per_column"].append({
            "column": col,
            "missing_count": missing_count,
            "missing_pct": missing_pct,
            "priority": priority,
            "status": status,
            "action": action
        })
        
        # Store example rows ONLY for columns with missing values
        if missing_count > 0:
            missing_rows = raw[col_missing[col]].head(5)
            display_cols = [c for c in ["Period", "Commodity", "Province", "Country", "Value ($)", "Quantity", "_source_file"] 
                           if c in missing_rows.columns]
            results["examples"][col] = missing_rows[display_cols].copy()
    
    # ═══════════════════════════════════════════════════════════
    # 2. PER ROW SUMMARY (completeness bands)
    # ═══════════════════════════════════════════════════════════
    row_miss_pct = (col_missing.sum(axis=1) / len(df.columns) * 100).round(1)
    
    bands = [
        ("100% complete", (row_miss_pct == 0).sum()),
        ("1–25% missing", ((row_miss_pct > 0) & (row_miss_pct <= 25)).sum()),
        ("26–50% missing", ((row_miss_pct > 25) & (row_miss_pct <= 50)).sum()),
        (">50% missing", (row_miss_pct > 50).sum()),
    ]
    
    for band_name, count in bands:
        pct_of_total = (count / n * 100).round(2) if n > 0 else 0
        status = "✅ OK" if band_name == "100% complete" else "⚠️ WARN" if count > 0 else "✅ OK"
        
        results["per_row_summary"].append({
            "band": band_name,
            "row_count": int(count),
            "pct_of_total": pct_of_total,
            "status": status
        })
    
    # Store example rows with >50% missing
    if (row_miss_pct > 50).any():
        high_missing_rows = raw[row_miss_pct > 50].head(5)
        display_cols = [c for c in df.columns if c in high_missing_rows.columns]
        results["examples"]["rows_with_high_missing"] = high_missing_rows[display_cols + ["_source_file"]].copy()
    
    # ═══════════════════════════════════════════════════════════
    # 3. MISSING KEY IDENTIFIERS
    # ═══════════════════════════════════════════════════════════
    for col in KEY_COLS:
        if col in col_missing.columns:
            missing_count = int(col_missing[col].sum())
            missing_pct = round(col_missing[col].mean() * 100, 2)
            severity = "🔴 Critical" if missing_count > 0 else "✅ OK"
            
            results["key_identifiers"].append({
                "key_field": col,
                "missing_count": missing_count,
                "missing_pct": missing_pct,
                "severity": severity
            })
    
    # ═══════════════════════════════════════════════════════════
    # 4. PATTERN ANALYSIS (systematic missing values?)
    # ═══════════════════════════════════════════════════════════
    pattern_analysis = []
    
    for col in ["Commodity", "Province", "Country", "Value ($)", "Quantity"]:
        if col in df.columns and col_missing[col].any():
            # Group by Country and Province to find systematic patterns
            missing_mask = col_missing[col]
            if missing_mask.sum() > 0:
                grouped = (
                    raw[missing_mask]
                    .groupby(["Country", "Province"], dropna=False)
                    .size()
                    .reset_index(name="missing_count")
                    .sort_values("missing_count", ascending=False)
                    .head(5)
                )
                
                for _, r in grouped.iterrows():
                    pattern_type = "Systematic" if r["missing_count"] > 1 else "Random"
                    recommendation = "Investigate source system" if r["missing_count"] > 5 else "Spot check"
                    
                    pattern_analysis.append({
                        "missing_column": col,
                        "country": r.get("Country", "-"),
                        "province": r.get("Province", "-"),
                        "missing_count": int(r["missing_count"]),
                        "pattern_type": pattern_type,
                        "recommendation": recommendation
                    })
    
    results["patterns"] = pattern_analysis
    
    return results

# ──────────────────────────────────────────────────────────────
# PRINT RESULTS
# ──────────────────────────────────────────────────────────────
def print_results(results):
    print("\n" + "="*80)
    print("  MODULE 2 — MISSING VALUES ANALYSIS")
    print("="*80)
    
    # ─────────────────────────────────────────────────────────
    # SECTION 1: Per Column
    # ─────────────────────────────────────────────────────────
    print("\n" + "─"*80)
    print("  1. MISSING VALUES PER COLUMN")
    print("─"*80 + "\n")
    
    for i, r in enumerate(results["per_column"], 1):
        print(f"{i}. {r['column']}")
        print(f"   Missing:  {r['missing_count']:,} rows ({r['missing_pct']}%)")
        print(f"   Priority: {r['priority']}")
        print(f"   Status:   {r['status']}")
        print(f"   Action:   {r['action']}")
        
        # Print example rows if available
        col_key = r["column"]
        if col_key in results["examples"] and r['missing_count'] > 0:
            print(f"\n   📋 Example rows with missing {r['column']} (showing up to 5):")
            print("   " + "-" * 76)
            df_example = results["examples"][col_key]
            
            for idx, row in df_example.iterrows():
                print(f"   Row {idx}:")
                for col in df_example.columns:
                    val = row[col]
                    if pd.isna(val):
                        val_str = "[MISSING]"
                    elif str(val).strip() == "":
                        val_str = "[EMPTY]"
                    else:
                        val_str = str(val)[:50]
                    print(f"      {col:20s}: {val_str}")
                print()
            print("   " + "-" * 76)
        
        print()
    
    # ─────────────────────────────────────────────────────────
    # SECTION 2: Per Row Summary
    # ─────────────────────────────────────────────────────────
    print("\n" + "─"*80)
    print("  2. ROW COMPLETENESS SUMMARY")
    print("─"*80 + "\n")
    
    for i, r in enumerate(results["per_row_summary"], 1):
        print(f"{i}. {r['band']}")
        print(f"   Row count:   {r['row_count']:,} ({r['pct_of_total']}% of total)")
        print(f"   Status:      {r['status']}")
        print()
    
    # Show example of highly incomplete rows
    if "rows_with_high_missing" in results["examples"]:
        print("\n   📋 Example rows with >50% missing data (showing up to 5):")
        print("   " + "-" * 76)
        df_example = results["examples"]["rows_with_high_missing"]
        
        for idx, row in df_example.iterrows():
            print(f"   Row {idx}:")
            missing_cols = []
            for col in df_example.columns:
                val = row[col]
                if pd.isna(val) or str(val).strip() == "":
                    missing_cols.append(col)
            print(f"      Missing columns: {', '.join(missing_cols) if missing_cols else 'None'}")
            
            # Show non-missing values
            for col in df_example.columns:
                val = row[col]
                if not pd.isna(val) and str(val).strip() != "":
                    val_str = str(val)[:50]
                    print(f"      {col:20s}: {val_str}")
            print()
        print("   " + "-" * 76 + "\n")
    
    # ─────────────────────────────────────────────────────────
    # SECTION 3: Missing Key Identifiers
    # ─────────────────────────────────────────────────────────
    print("\n" + "─"*80)
    print("  3. MISSING KEY IDENTIFIERS")
    print("─"*80 + "\n")
    
    for i, r in enumerate(results["key_identifiers"], 1):
        print(f"{i}. {r['key_field']}")
        print(f"   Missing:  {r['missing_count']:,} rows ({r['missing_pct']}%)")
        print(f"   Severity: {r['severity']}")
        print()
    
    # ─────────────────────────────────────────────────────────
    # SECTION 4: Pattern Analysis
    # ─────────────────────────────────────────────────────────
    print("\n" + "─"*80)
    print("  4. MISSING VALUE PATTERNS (Top Clusters)")
    print("─"*80 + "\n")
    
    if results["patterns"]:
        for i, r in enumerate(results["patterns"], 1):
            print(f"{i}. Missing column: {r['missing_column']}")
            print(f"   Province:       {r['province']}")
            print(f"   Missing count:  {r['missing_count']}")
            print(f"   Pattern type:   {r['pattern_type']}")
            print(f"   Action:         {r['recommendation']}")
            print()
    else:
        print("   No systematic patterns detected (all missing values appear random)\n")

# ──────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "="*80)
    print("  MISSING VALUES ANALYSIS — MODULE 2 ONLY")
    print("="*80)

    export_files = glob.glob(str(EXPORTS_DIR / "**/*.csv"), recursive=True)
    print(f"\nFound {len(export_files)} CSV file(s) in {EXPORTS_DIR}")

    if not export_files:
        print("❌ No files found. Check EXPORTS_DIR path.")
        exit(1)

    print("Loading files...")
    raw, load_errors = load_all(export_files)
    print(f"Loaded {len(raw):,} rows from {len(set(raw['_source_file']))} file(s)")
    if load_errors:
        print(f"⚠️  {len(load_errors)} file(s) failed to load")

    print("\nRunning missing values analysis...")
    results = module_missing(raw)
    
    print_results(results)
    
    # Summary
    total_cols_with_missing = sum(1 for r in results["per_column"] if r["missing_count"] > 0)
    critical_missing = sum(1 for r in results["key_identifiers"] if r["missing_count"] > 0)
    
    print("="*80)
    print(f"  SUMMARY:")
    print(f"    Columns with missing data: {total_cols_with_missing}/{len(results['per_column'])}")
    print(f"    Critical (key identifiers): {critical_missing}/{len(results['key_identifiers'])}")
    print(f"    Systematic patterns found: {len(results['patterns'])}")
    print("="*80 + "\n")


  MISSING VALUES ANALYSIS — MODULE 2 ONLY

Found 154 CSV file(s) in /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/Exports
Loading files...
Loaded 2,536,465 rows from 154 file(s)

Running missing values analysis...

  MODULE 2 — MISSING VALUES ANALYSIS

────────────────────────────────────────────────────────────────────────────────
  1. MISSING VALUES PER COLUMN
────────────────────────────────────────────────────────────────────────────────

1. Period
   Missing:  0 rows (0.0%)
   Priority: ✅ NONE
   Status:   ✅ OK
   Action:   None

2. Commodity
   Missing:  0 rows (0.0%)
   Priority: ✅ NONE
   Status:   ✅ OK
   Action:   None

3. Province
   Missing:  0 rows (0.0%)
   Priority: ✅ NONE
   Status:   ✅ OK
   Action:   None

4. Country
   Missing:  0 rows (0.0%)
   Priority: ✅ NONE
   Status:   ✅ OK
   Action:   None

5. State
   Missing:  816,179 rows (32.18%)
   Priority: 🟢 LOW
   Status:   ⚠️ WARN
   Action:   Optional — low priority cleanup

   📋 Example rows with missin